# Workflow 2: Hierarchical Agentic RAG (TH4)

In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

import time
import re
import requests
import json
from typing import List, Dict, Any

import uuid
import operator
from typing import TypedDict, Literal, Optional, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from configs.setting import settings
from configs.GetConfig import config

from src.LLMService import LLMService
from src.e_agents.guardrail_call import GuardrailCall
from src.e_agents.rejection_call import RejectionCall

from src.d_tools import (
    product_search, 
    product_compare,
    policy_search,
    order_lookup,
)

from src.d_tools import (
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
)

from src.f_prompts import (
    FULL_MASTER_PROMPT,        
    FULL_REJECTION_PROMPT
)

from app.core.security import verify_supabase_jwt

from src.f_prompts.skills import load_skill, list_skills


In [2]:
available_tools = {
    "product_search": product_search,
    "product_compare": product_compare,
    "policy_search": policy_search,
    "order_lookup": order_lookup
}

tools_schema = [
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
]

# Cấu hình từng domain node: tool nào và schema nào được phép trong node đó
DOMAIN_CONFIGS = {
    "product": {
        "available": {
            "product_search": product_search,
            "product_compare": product_compare,
        },
        "schemas": [PRODUCT_SEARCH_SCHEMA, PRODUCT_COMPARE_SCHEMA],
    },
    "policy": {
        "available": {"policy_search": policy_search},
        "schemas": [POLICY_SEARCH_SCHEMA],
    },
    "account": {
        "available": {"order_lookup": order_lookup},
        "schemas": [ORDER_LOOKUP_SCHEMA],
    },
}

# ============================================================================
# ROUTING SCHEMAS: Master Agent chỉ nhìn thấy 3 meta-tool chọn domain
# ============================================================================
ROUTE_TO_PRODUCT_SCHEMA = {
    "type": "function",
    "function": {
        "name": "route_to_product",
        "description": "Chuyển câu hỏi liên quan sản phẩm (tìm kiếm, so sánh, tồn kho, giá) sang bộ phận Sản phẩm xử lý.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }
}
ROUTE_TO_POLICY_SCHEMA = {
    "type": "function",
    "function": {
        "name": "route_to_policy",
        "description": "Chuyển câu hỏi liên quan chính sách (đổi trả, bảo hành, vận chuyển) sang bộ phận Chính sách xử lý.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }
}
ROUTE_TO_ACCOUNT_SCHEMA = {
    "type": "function",
    "function": {
        "name": "route_to_account",
        "description": "Chuyển câu hỏi liên quan đơn hàng/tài khoản cá nhân của khách sang bộ phận Tài khoản xử lý.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }
}
ROUTING_SCHEMAS = [ROUTE_TO_PRODUCT_SCHEMA, ROUTE_TO_POLICY_SCHEMA, ROUTE_TO_ACCOUNT_SCHEMA]


In [3]:
# ============================================================================
# Helpers: parse Gemini streaming response + sanitize tool arguments
# ============================================================================

AUTH_TOOLS = ["order_lookup", "cart_lookup", "wishlist_update"]


def _try_json(s):
    """Thử parse chuỗi JSON; nếu lỗi trả về None."""
    if not isinstance(s, str):
        return None
    try:
        return json.loads(s)
    except Exception:
        return None


def _sanitize_tool_args(name, args):
    """Chuẩn hóa args của tool trước khi gọi hàm thật."""
    if isinstance(args, str):
        parsed = _try_json(args)
        args = parsed if isinstance(parsed, dict) else {"queries": [args]}
    elif isinstance(args, list):
        args = {"queries": args}
    elif not isinstance(args, dict):
        args = {}

    if name == "product_search":
        allowed_keys = {"keyword", "brand", "category", "min_price", "max_price",
                        "name_contains", "mode", "limit", "include_details", "need_price_info"}
        if "queries" not in args:
            top = {k: v for k, v in args.items() if k in allowed_keys}
            args = {"queries": [top] if top else []}
        queries = args["queries"]
        if isinstance(queries, str):
            parsed = _try_json(queries)
            queries = parsed if isinstance(parsed, list) else [queries]
        if not isinstance(queries, list):
            queries = [queries]
        clean_queries = []
        for q in queries:
            if isinstance(q, str):
                parsed = _try_json(q)
                q = parsed if isinstance(parsed, dict) else {"keyword": q}
            if not isinstance(q, dict):
                continue
            if q.get("limit") is None:
                q["limit"] = 30 if q.get("mode") == "lines" else 3
            if not q.get("keyword"):
                q["keyword"] = f"{q.get('brand', '')} {q.get('category', '')}".strip() or "sản phẩm"
            nc = q.get("name_contains")
            if nc is not None and len(str(nc).strip()) <= 2:
                q["name_contains"] = q.get("keyword")
            clean_queries.append({k: v for k, v in q.items() if k in allowed_keys})
        return {"queries": clean_queries}

    if name == "product_compare":
        names = args.get("product_names") or args.get("products") or args.get("product_name") or []
        if isinstance(names, str):
            names = [names]
        if not isinstance(names, list):
            names = []
        return {"product_names": [n for n in names if isinstance(n, str)]}

    if name == "policy_search":
        kw = args.get("key_word", "") or args.get("keyword", "")
        return {"key_word": kw, "limit": args.get("limit", 3)}

    if name == "order_lookup":
        out = {}
        if "order_id" in args:
            out["order_id"] = args["order_id"]
        return out

    return args


def _parse_gemini_response(response_stream):
    """Trích xuất text, tool_calls, tokens từ stream của Gemini."""
    text_content = ""
    tool_calls = {}
    input_tokens = 0
    output_tokens = 0
    for chunk in response_stream:
        if getattr(chunk, "usage_metadata", None):
            input_tokens = getattr(chunk.usage_metadata, "prompt_token_count", 0) or 0
            output_tokens = getattr(chunk.usage_metadata, "candidates_token_count", 0) or 0

        if getattr(chunk, "candidates", None) and chunk.candidates:
            for p in chunk.candidates[0].content.parts:
                txt = getattr(p, "text", None)
                if txt:
                    text_content += txt

                fc = getattr(p, "function_call", None)
                if fc:
                    call_id = getattr(fc, "id", None) or f"call_{len(tool_calls)}"
                    args = fc.args
                    if not isinstance(args, dict):
                        args = dict(args) if hasattr(args, "keys") else {}
                    sig = getattr(p, "thought_signature", None)
                    tool_calls[call_id] = {
                        "id": call_id,
                        "name": fc.name,
                        "args": args,
                        "thought_signature": sig,
                    }
        else:
            if getattr(chunk, "text", None):
                text_content += chunk.text
            calls = getattr(chunk, "function_calls", None)
            if calls:
                for call in calls:
                    call_id = getattr(call, "id", None) or f"call_{len(tool_calls)}"
                    args = call.args
                    if not isinstance(args, dict):
                        args = dict(args) if hasattr(args, "keys") else {}
                    tool_calls[call_id] = {
                        "id": call_id,
                        "name": call.name,
                        "args": args,
                        "thought_signature": None,
                    }
    return text_content, list(tool_calls.values()), input_tokens, output_tokens


def _last_is_model(msgs):
    if not msgs:
        return False
    last = msgs[-1]
    if isinstance(last, dict):
        return last.get("role") in ("assistant", "model")
    return getattr(last, "type", None) in ("ai", "assistant")


# ============================================================================
# MasterAgent (Router): chỉ quyết định domain tiếp theo, KHÔNG thực thi tool
# ============================================================================

class MasterAgent:
    def __init__(self, llm_service: LLMService, config):
        self.llm_service = llm_service
        self.config = config
        self.model = config.llm.google.available[0]

    def invoke(self, messages, tools_schema):
        start_time = time.time()
        # Gemini API không cho phép request kết thúc bằng model turn.
        # Nếu history dẫn đến assistant cuối (ví dụ sau domain node), chèn user prompt nội bộ.
        call_messages = messages
        if _last_is_model(call_messages):
            call_messages = list(call_messages) + [{
                "role": "user",
                "content": "Tiếp tục phân tích và quyết định bước tiếp theo."
            }]
        response = self.llm_service.call_gemini(
            model=self.model,
            messages=call_messages,
            tools=tools_schema,
            stream=True
        )
        text_content, raw_calls, input_tokens, output_tokens = _parse_gemini_response(response)

        tool_calls = []
        for i, c in enumerate(raw_calls):
            call_id = c["id"] or f"call_router_{i}"
            tool_calls.append({
                "id": call_id,
                "type": "function",
                "thought_signature": c.get("thought_signature"),
                "function": {
                    "name": c["name"],
                    "arguments": json.dumps(c["args"], ensure_ascii=False)
                }
            })

        latency = time.time() - start_time
        return {
            "content": text_content if text_content else None,
            "tool_calls": tool_calls,
            "tokens": {
                "input": input_tokens,
                "output": output_tokens,
                "total": input_tokens + output_tokens,
            },
            "latency": latency,
        }


# ============================================================================
# DomainAgent: ReAct nội bộ trong 1 domain, tự chọn tool + tham số
# ============================================================================

class DomainAgent:
    def __init__(self, llm_service: LLMService, config):
        self.llm_service = llm_service
        self.config = config
        self.model = config.llm.google.available[0]

    def _inject_skill(self, messages, skill_text):
        if not skill_text:
            return messages
        messages = list(messages)
        insert_at = 0
        for i, m in enumerate(messages):
            if isinstance(m, dict) and m.get("role") == "system":
                insert_at = i + 1
                break
        messages.insert(insert_at, {"role": "system", "content": skill_text})
        return messages

    def invoke(self, messages, available_tools, tools_schema, auth_context=None, max_turns=None, skill=None):
        messages = list(messages)
        if skill:
            messages = self._inject_skill(messages, skill)
        start_time = time.time()
        input_tokens = 0
        output_tokens = 0
        tool_context = []
        max_turns = max_turns or self.config.agent.max_turns

        for turn in range(max_turns):
            # Nếu messages kết thúc bằng assistant (ví dụ lượt trước domain khác trả về),
            # chèn user prompt nội bộ để Gemini chấp nhận, nhưng KHÔNG lưu vào messages.
            call_messages = messages
            if _last_is_model(call_messages):
                call_messages = list(call_messages) + [{
                    "role": "user",
                    "content": "Tiếp tục xử lý trong domain này."
                }]

            response = self.llm_service.call_gemini(
                model=self.model,
                messages=call_messages,
                tools=tools_schema,
                stream=True
            )
            text_content, tool_calls, inp, out = _parse_gemini_response(response)
            input_tokens += inp
            output_tokens += out

            if tool_calls:
                assistant_msg = {
                    "role": "assistant",
                    "content": text_content if text_content else None,
                    "tool_calls": [
                        {
                            "id": c["id"],
                            "type": "function",
                            "thought_signature": c.get("thought_signature"),
                            "function": {
                                "name": c["name"],
                                "arguments": json.dumps(c["args"], ensure_ascii=False)
                            }
                        }
                        for c in tool_calls
                    ]
                }
                messages.append(assistant_msg)

                for c in tool_calls:
                    func_name = c["name"]
                    func_args = _sanitize_tool_args(func_name, c["args"])
                    if func_name in AUTH_TOOLS and auth_context:
                        func_args["current_user_id"] = auth_context.get("user_id")
                        func_args["user_token"] = auth_context.get("user_token")

                    if func_name in available_tools:
                        try:
                            result = available_tools[func_name](**func_args)
                        except Exception as e:
                            result = f"Lỗi thực thi {func_name}: {e}"
                    else:
                        result = f"Tool '{func_name}' không khả dụng trong domain này."

                    messages.append({
                        "role": "tool",
                        "tool_call_id": c["id"],
                        "name": func_name,
                        "content": str(result)
                    })
                    tool_context.append({"tool": func_name, "args": func_args, "output": str(result)})
                continue
            else:
                if text_content:
                    messages.append({"role": "assistant", "content": text_content})
                break

        latency = time.time() - start_time
        return {
            "content": text_content,
            "messages": messages,
            "tool_context": tool_context,
            "tokens": {
                "input": input_tokens,
                "output": output_tokens,
                "total": input_tokens + output_tokens,
            },
            "latency": latency,
        }


In [4]:
llm_service = LLMService(settings, config)
guardrail_call = GuardrailCall(llm_service, config)
rejection_call = RejectionCall(llm_service, config)
master_agent = MasterAgent(llm_service, config)

print("Available skills:", list_skills())


Available skills: ['account/order_lookup', 'common/multi_turn_context', 'policy/policy_search', 'product/ambiguous', 'product/single_spec']


In [5]:
class RetrievedChunk(TypedDict):
    content: str
    source: str
    score: float
    chunk_type: str

class AgentState(TypedDict):

    # 1. INput user
    user_query: str
    session_id: str

    # 2. Auth
    user_token: Optional[str]
    user_id: Optional[str]
    is_authenticated: bool

    # 3. Guardrail & Quality
    risk_level: Optional[str]
    relevance_score: float

    # 4. Router
    next_domains: list[str]

    # 5. Retrieval & Tools
    retrieved_context: list[RetrievedChunk]
    tool_calls_used: Annotated[list[dict], operator.add]
    iteration_count: int

    # 6. Hội thoại
    messages: Annotated[list, operator.add]
    conversation_state: dict

    # 7. Output
    final_answer: Optional[str]
    cited_sources: list[str]
    ticket_id: Optional[str]
    show_popup: bool

    # 8. Thống kê
    input_tokens: Annotated[int, operator.add]
    output_tokens: Annotated[int, operator.add]
    latency: Annotated[float, operator.add]
    total_tokens: Annotated[int, operator.add]


In [6]:
def receive_node(state: AgentState) -> dict:
    query = state.get("user_query", "").strip()
    token = state.get("user_token")
    user_id = None

    if token:
        user_id = verify_supabase_jwt(token)
    
    is_authenticated = True if user_id else False
    
    return {
        "user_id": user_id,
        "is_authenticated": is_authenticated,
        "user_query": query  
    }


In [7]:
def guardrail_node(state: AgentState) -> dict:
    query = state["user_query"]
    
    result = guardrail_call.invoke(query)
    risk_level = result["risk_level"]
    
    if risk_level != "attack":
        return {
            "risk_level": risk_level,
            "show_popup": False,
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ],
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }
    else:
        return {
            "risk_level": risk_level,
            "show_popup": True,
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }


In [8]:
MAX_OUTER_TURNS = config.agent.max_turns


def _first_message_is_system(msgs):
    if not msgs:
        return False
    first = msgs[0]
    if isinstance(first, dict):
        return first.get("role") == "system"
    return getattr(first, "type", None) == "system"


ROUTER_INSTRUCTION = (
    "Bạn là ROUTER của cửa hàng. Nhiệm vụ DUY NHẤT là quyết định câu hỏi cần chuyển đến domain nào.\n"
    "- Sử dụng các tool 'route_to_product', 'route_to_policy', 'route_to_account' để chuyển domain.\n"
    "- Có thể chọn NHIỀU domain trong 1 lượt nếu câu hỏi kết hợp nhiều lĩnh vực (ví dụ vừa hỏi tồn kho vừa hỏi chính sách).\n"
    "- Nếu đã có đủ thông tin trong lịch sử hội thoại để trả lời, KHÔNG gọi tool, hãy trả lời trực tiếp.\n"
    "- Khi trả lời trực tiếp, giữ văn phong nhân viên CSKH, ngắn gọn, tự nhiên."
)


def master_node(state: AgentState) -> dict:
    """
    Master Agent đóng vai trò Router:
      - Gọi LLM với ROUTING_SCHEMAS (chỉ 3 meta-tool chọn domain).
      - Nếu trả về text -> đây là câu trả lời cuối.
      - Nếu trả về tool_calls (route_to_*) -> set next_domains để graph chuyển đến domain node.
      - Domain node sẽ tự chọn tool cụ thể + tham số + ReAct nội bộ.
    """
    if state.get("final_answer"):
        return {}

    if state.get("iteration_count", 0) >= MAX_OUTER_TURNS:
        return {"final_answer": "Đã đạt giới hạn lượt suy luận ngoài."}

    msgs = list(state["messages"])
    if not _first_message_is_system(msgs):
        msgs = [{"role": "system", "content": f"{FULL_MASTER_PROMPT}\n\n{ROUTER_INSTRUCTION}"}] + msgs
    else:
        msgs = [msgs[0]] + msgs[1:]

    res = master_agent.invoke(messages=msgs, tools_schema=ROUTING_SCHEMAS)

    updates = {
        "input_tokens": res["tokens"]["input"],
        "output_tokens": res["tokens"]["output"],
        "total_tokens": res["tokens"]["total"],
        "latency": res["latency"],
        "iteration_count": state.get("iteration_count", 0) + 1,
        "next_domains": [],
    }

    if res["tool_calls"]:
        mapping = {
            "route_to_product": "product",
            "route_to_policy": "policy",
            "route_to_account": "account",
        }
        domains = []
        for c in res["tool_calls"]:
            name = c["function"]["name"]
            if name in mapping and mapping[name] not in domains:
                domains.append(mapping[name])
        updates["next_domains"] = domains
    else:
        if res["content"]:
            updates["messages"] = [{"role": "assistant", "content": res["content"]}]
            updates["final_answer"] = res["content"]
        else:
            updates["final_answer"] = "Xin lỗi, tôi chưa thể xử lý yêu cầu này."

    return updates


In [9]:
def rejection_node(state: AgentState) -> dict:
    """
    [NODE] Rejection Agent (Từ chối):
    - Chỉ chạy khi risk_level == "needs_ticket"
    - Trả về câu từ chối lịch sự
    - KHÔNG lưu tin nhắn vào messages (đã lưu ở guardrail_node)
    """
    
    query = state["user_query"]

    result = rejection_call.invoke(query)

    return {
        "final_answer": result["content"],
        "show_popup": True,
        "input_tokens": result["tokens"]["input"],
        "output_tokens": result["tokens"]["output"],
        "total_tokens": result["tokens"]["input"] + result["tokens"]["output"],
        "latency": result["latency"]
    }


In [10]:
def _domain(call: dict) -> str:
    """Xác định domain từ route meta-tool MasterAgent đã gọi."""
    name = call["function"]["name"]
    mapping = {"route_to_product": "product", "route_to_policy": "policy", "route_to_account": "account"}
    return mapping.get(name, "")


def _select_skill_for_domain(query: str, domain: str) -> str:
    """Chọn skill markdown phù hợp với domain và câu hỏi."""
    q = (query or "").lower()
    if domain == "account":
        return load_skill("account/order_lookup") or ""
    if domain == "policy":
        return load_skill("policy/policy_search") or ""
    if domain == "product":
        if any(k in q for k in ["so sánh", "nên chọn", "hay hơn"]):
            return load_skill("product/compare") or load_skill("product/single_spec") or ""
        if any(k in q for k in ["bao nhiêu", "giá", "tồn kho", "chip", "ram", "pin", "màn hình", "camera", "bộ nhớ"]):
            return load_skill("product/single_spec") or ""
        return load_skill("product/ambiguous") or ""
    return ""


def _detect_status(tool_context: list, content: str = "") -> str:
    """Xác định status của domain dựa trên kết quả tool và nội dung trả lời."""
    if not tool_context:
        return "found" if content else "not_found"
    NOT_FOUND_MARKERS = [
        "no matching products found",
        "no matching store policies found",
        "no information found in the database",
        "please provide product names to compare",
        "you have not placed any orders yet",
        "no order found",
        "the user is currently unauthenticated",
    ]
    outputs = " ".join(str(t.get("output", "")).lower() for t in tool_context)
    found_markers = [m for m in NOT_FOUND_MARKERS if m in outputs]
    if not found_markers:
        return "found"
    if len(tool_context) == 1 or len(found_markers) == len(tool_context):
        return "not_found"
    return "partial"


def _build_domain_system_prompt(domain: str, available: dict) -> str:
    """System prompt riêng cho từng domain."""
    allowed_tools = list(available.keys())
    base = (
        f"{FULL_MASTER_PROMPT}\n\n"
        f"Bạn đang xử lý domain '{domain}'. Bạn chỉ được phép sử dụng các tool sau: {allowed_tools}.\n"
        f"Nếu câu hỏi có thể trả lời trực tiếp từ thông tin đã có trong lịch sử hội thoại (kết quả tra cứu trước đó), "
        f"hãy trả lời ngay, không cần gọi tool mới.\n"
    )
    if domain == "product":
        base += (
            "\nKhi câu hỏi có điều kiện 'hoặc' (ví dụ 'Asus hoặc Lenovo RAM 16GB'), "
            "hãy tách thành NHIỀU object queries trong MỘT lần gọi product_search, không gộp chung 1 keyword.\n"
            "SAI: {\"queries\": [{\"keyword\": \"laptop Asus hoặc Lenovo RAM 16GB\"}]}\n"
            "ĐÚNG: {\"queries\": [{\"keyword\": \"laptop Asus RAM 16GB\"}, {\"keyword\": \"laptop Lenovo RAM 16GB\"}]}\n"
            "Khi khách hỏi tiếp về cùng nhóm sản phẩm đã tra cứu, hãy tái sử dụng brand/category/name_contains/mode từ lượt trước, "
            "chỉ điều chỉnh max_price/min_price/limit/include_details/need_price_info theo yêu cầu mới."
        )
    if domain == "account":
        base += "\nNếu user chưa xác thực (user_token trống), hãy yêu cầu đăng nhập trước khi tra cứu đơn hàng."
    return base


def _run_domain_node(state: AgentState, domain: str, available: dict, schemas: list) -> dict:
    """Gọi DomainAgent để tự chọn tool + tham số + ReAct nội bộ trong domain."""
    history = list(state["messages"])
    sys_content = _build_domain_system_prompt(domain, available)
    skill = _select_skill_for_domain(state.get("user_query", ""), domain)
    if skill:
        sys_content += "\n\n" + skill
    sys_msg = {"role": "system", "content": sys_content}
    msgs = [sys_msg] + history

    auth = {"user_id": state.get("user_id"), "user_token": state.get("user_token")}

    agent = DomainAgent(llm_service, config)
    res = agent.invoke(
        msgs,
        available_tools=available,
        tools_schema=schemas,
        auth_context=auth,
        max_turns=config.agent.max_turns,
    )

    new_messages = res["messages"][len(msgs):]

    status = _detect_status(res.get("tool_context", []), res.get("content", ""))
    tool_calls_used = res.get("tool_context", []) + [{"tool": "_domain_status", "domain": domain, "status": status}]

    return {
        "messages": new_messages,
        "tool_calls_used": tool_calls_used,
        "input_tokens": res["tokens"]["input"],
        "output_tokens": res["tokens"]["output"],
        "total_tokens": res["tokens"]["total"],
        "latency": res["latency"],
    }


def account_node(state: AgentState) -> dict:
    return _run_domain_node(
        state, "account",
        DOMAIN_CONFIGS["account"]["available"],
        DOMAIN_CONFIGS["account"]["schemas"]
    )


In [11]:
def product_node(state: AgentState) -> dict:
    return _run_domain_node(
        state, "product",
        DOMAIN_CONFIGS["product"]["available"],
        DOMAIN_CONFIGS["product"]["schemas"]
    )


In [12]:
def policy_node(state: AgentState) -> dict:
    return _run_domain_node(
        state, "policy",
        DOMAIN_CONFIGS["policy"]["available"],
        DOMAIN_CONFIGS["policy"]["schemas"]
    )


In [13]:
def route_guardrail(state: AgentState):
    """Điều hướng sau guardrail: attack/needs_ticket -> rejection, safe -> master."""
    if state.get("risk_level") in ("attack", "needs_ticket"):
        return "rejection"
    return "master"


def route_master(state: AgentState):
    """Điều hướng sau MasterAgent router: fan-out đến các domain trong next_domains."""
    if state.get("final_answer"):
        return END

    domains = state.get("next_domains", [])
    if domains:
        return domains
    return END


def route_after_domain(state: AgentState):
    """Sau mỗi domain node, LUÔN quay lại master để quyết định bước tiếp theo."""
    if state.get("final_answer"):
        return END
    return "master"


# ============================================================================
# Build & compile graph
# ============================================================================

workflow = StateGraph(AgentState)
workflow.add_node("receive", receive_node)
workflow.add_node("guardrail", guardrail_node)
workflow.add_node("master", master_node)
workflow.add_node("product", product_node)
workflow.add_node("policy", policy_node)
workflow.add_node("account", account_node)
workflow.add_node("rejection", rejection_node)

workflow.add_edge(START, "receive")
workflow.add_edge("receive", "guardrail")
workflow.add_conditional_edges("guardrail", route_guardrail)
workflow.add_conditional_edges("master", route_master)
workflow.add_conditional_edges("product", route_after_domain)
workflow.add_conditional_edges("policy", route_after_domain)
workflow.add_conditional_edges("account", route_after_domain)
workflow.add_edge("rejection", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
print("✅ Workflow 2 (TH4) compiled successfully.")


✅ Workflow 2 (TH4) compiled successfully.


In [14]:
# ============================================================================
# TEST 1: Một câu hỏi đơn giản về laptop văn phòng dưới 30 triệu
# ============================================================================

user_token = ""  # Thay bằng JWT token nếu cần test order_lookup / auth

run_config = {"configurable": {"thread_id": "session_test_notebook_002"}}

res = app.invoke(
    {
        "user_query": "Em ơi, bên em có mã laptop văn phòng nào dưới 30 triệu không, tư vấn chị vài mẫu với.",
        "user_token": user_token,
    },
    config=run_config,
)

print()
print("📊 BÁO CÁO THỐNG KÊ CHI TIẾT (AGENT STATE METRICS)")
print("=" * 60)
print(f"💬 Câu trả lời (Final Answer) : {res.get('final_answer')}")
print(f"🛡️ Mức độ rủi ro (Risk Level) : {res.get('risk_level')}")
print(f"⏱️ Tổng độ trễ (Total Latency): {res.get('latency', 0):.2f}s")
print(f"📥 Input Tokens               : {res.get('input_tokens', 0)}")
print(f"📤 Output Tokens              : {res.get('output_tokens', 0)}")
print(f"🧮 Tổng Tokens (Total Tokens) : {res.get('total_tokens', 0)}")

print("\n📜 LỊCH SỬ HỘI THOẠI (MESSAGES HISTORY):")
for idx, msg in enumerate(res.get("messages", []), 1):
    role = getattr(msg, "type", None) or (msg.get("role") if isinstance(msg, dict) else "unknown")
    content = getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, dict) else "")
    print(f"  [{idx}] {role.upper()}: {content}")

print("=" * 60)



📊 BÁO CÁO THỐNG KÊ CHI TIẾT (AGENT STATE METRICS)
💬 Câu trả lời (Final Answer) : Dạ chị muốn em tư vấn chi tiết hơn về cấu hình, dung lượng RAM hay tính năng của mẫu nào trong số các dòng trên ạ?
🛡️ Mức độ rủi ro (Risk Level) : safe
⏱️ Tổng độ trễ (Total Latency): 15.44s
📥 Input Tokens               : 18370
📤 Output Tokens              : 278
🧮 Tổng Tokens (Total Tokens) : 18648

📜 LỊCH SỬ HỘI THOẠI (MESSAGES HISTORY):
  [1] USER: Em ơi, bên em có mã laptop văn phòng nào dưới 30 triệu không, tư vấn chị vài mẫu với.
  [2] ASSISTANT: None
  [3] TOOL: ["văn phòng"]:
Product: Laptop ASUS VivoBook X415EA-EK2043W | ID: 66282
- ID: 66282
- SKU: laptop-asus-vivobook-x415ea-ek2043w
- Out of Stock
- Price: 9,290,000 VND | Original: 11,490,000 VND | Discount: 19.15%
- Brand: ASUS
- Score: 0.0264

Product: Laptop Asus VivoBook Flip 14 TM420UA-EC182W | ID: 44505
- ID: 44505
- SKU: laptop-asus-vivobook-flip-14-tm420ua-ec182w
- Out of Stock
- Price: 17,990,000 VND | Original: 20,490,000 VND | Discoun

## Test 2: Truy vấn kết hợp nhiều domain (product + policy)

In [15]:
run_config2 = {"configurable": {"thread_id": "session_test_notebook_002_multi"}}

res2 = app.invoke(
    {
        "user_query": "Samsung S25 còn hàng không và chính sách đổi trả sao",
        "user_token": "",
    },
    config=run_config2,
)

print("\n📊 BÁO CÁO MULTI-DOMAIN")
print("=" * 60)
print(f"next_domains (lần đầu): {res2.get('next_domains', 'N/A')}")
print(f"💬 Câu trả lời (Final Answer) : {res2.get('final_answer')}")
print(f"🛡️ Risk Level : {res2.get('risk_level')}")
print(f"⏱️ Latency: {res2.get('latency', 0):.2f}s")
print(f"📥 Input Tokens: {res2.get('input_tokens', 0)} | 📤 Output Tokens: {res2.get('output_tokens', 0)} | 🧮 Total: {res2.get('total_tokens', 0)}")
print(f"🧰 Tool calls used count: {len(res2.get('tool_calls_used', []))}")
print("\n📜 MESSAGES HISTORY (compact):")
for idx, msg in enumerate(res2.get("messages", []), 1):
    role = getattr(msg, "type", None) or (msg.get("role") if isinstance(msg, dict) else "unknown")
    content = getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, dict) else "")
    if isinstance(content, str) and len(content) > 200:
        content = content[:200] + "..."
    print(f"  [{idx}] {role.upper()}: {content}")
print("=" * 60)



📊 BÁO CÁO MULTI-DOMAIN
next_domains (lần đầu): []
💬 Câu trả lời (Final Answer) : Dạ hiện em đã nắm được thông tin dòng Samsung S25 vẫn còn hàng và chính sách đổi trả trong vòng 30 ngày (với lỗi nhà sản xuất và còn nguyên hộp, phụ kiện) ạ. Anh/chị có cần em tư vấn thêm về giá, dung lượng hay thông tin chi tiết của mẫu nào không ạ?
🛡️ Risk Level : safe
⏱️ Latency: 24.04s
📥 Input Tokens: 25755 | 📤 Output Tokens: 417 | 🧮 Total: 26172
🧰 Tool calls used count: 4

📜 MESSAGES HISTORY (compact):
  [1] USER: Samsung S25 còn hàng không và chính sách đổi trả sao
  [2] ASSISTANT: None
  [3] TOOL: Policy Document 1 [Section: General Policy] (Type: FAQ, Score: 0.1820):
Tài liệu chính sách: Chính sách đổi trả 1-đổi-1 sản phẩm công nghệ
Nội dung chi tiết:
Chính sách đổi trả sản phẩm tại Website TM...
  [4] ASSISTANT: Dạ phần thông tin tồn kho của dòng Samsung S25 hiện tại em chưa có dữ liệu để kiểm tra chính xác cho anh/chị. 

Về chính sách đổi trả, bên em áp dụng đổi 1-đổi-1 trong vòng 30 ngày kể từ 

## Test 3: Truy vấn không tìm thấy kết quả

In [16]:
run_config3 = {"configurable": {"thread_id": "session_test_notebook_002_notfound"}}

res3 = app.invoke(
    {
        "user_query": "Tìm laptop gaming dưới 5 triệu",
        "user_token": "",
    },
    config=run_config3,
)

print("\n📊 BÁO CÁO NOT-FOUND")
print("=" * 60)
print(f"💬 Final Answer: {res3.get('final_answer')}")
print(f"🛡️ Risk Level : {res3.get('risk_level')}")
print(f"⏱️ Latency: {res3.get('latency', 0):.2f}s")
print(f"📥 Input Tokens: {res3.get('input_tokens', 0)} | 📤 Output Tokens: {res3.get('output_tokens', 0)} | 🧮 Total: {res3.get('total_tokens', 0)}")
print("=" * 60)



📊 BÁO CÁO NOT-FOUND
💬 Final Answer: Dạ anh/chị cần em tư vấn thêm về dòng sản phẩm nào hoặc có câu hỏi gì khác không ạ?
🛡️ Risk Level : safe
⏱️ Latency: 4.95s
📥 Input Tokens: 17907 | 📤 Output Tokens: 245 | 🧮 Total: 18152
